# Qwen3 Abstract Evaluator (Modular)

This notebook orchestrates reusable utilities from:
- `experiments/utils` (model-agnostic)
- `experiments/qwen/utils` (Qwen3-specific)

It supports:
- train/val/test load + clean/validate
- Qwen3 chat message construction
- JSONL export + HF datasets
- configurable eval every `epoch` or `steps`
- early stopping
- per-epoch JSON metrics artifacts
- post-training adapter eval + base-model eval
- W&B integration


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "experiments").exists() and (p / "data").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


PosixPath('/home/MohammadNabulsi/Essay Evaluator')

In [2]:
from experiments.qwen.utils.chat import add_messages_and_targets
from experiments.qwen.utils.configs import build_data_paths, build_qwen3_default_config
from experiments.qwen.utils.modeling import load_qwen3_model_for_inference
from experiments.qwen.utils.pipeline import (
    estimate_qwen_token_percentiles,
    evaluate_base_model,
    evaluate_saved_adapters,
    evaluate_single_adapter,
)
from experiments.qwen.utils.training import train_qwen3

from experiments.utils.data import (
    clean_train_val_test,
    load_train_val_test_dfs,
    score_distribution,
)
from experiments.utils.datasets_io import export_split_jsonl, to_hf_dataset_dict
from experiments.utils.evaluation import parse_rationale, parse_score
from experiments.utils.generation import generate_predictions_from_messages
from experiments.utils.logging_utils import setup_logger
from experiments.utils.runtime import configure_wandb_dir, set_global_seed



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[torchao|WARNING]Failed to load /home/MohammadNabulsi/Essay Evaluator/.venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/MohammadNabulsi/Essay Evaluator/.venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
[torchao|WARNING]Failed to load /home/MohammadNabulsi/Essay Evaluator/.venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/MohammadNabulsi/Essay Evaluator/.venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


In [3]:
cfg = build_qwen3_default_config(PROJECT_ROOT)

# --- Core run identity ---
cfg.model_name = "Qwen/Qwen3-8B"
cfg.run_name = "qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70"

# --- Data paths (edit if you want different splits) ---
cfg.data_paths = build_data_paths(
    train_path=PROJECT_ROOT / "data/data/train/all.jsonl",
    val_path=PROJECT_ROOT / "data/data/val/all.jsonl",
    test_path=PROJECT_ROOT / "data/data/test/all.jsonl",
)

# --- Trainer config (speed-first + train-until-no-improvement pattern) ---
cfg.max_seq_length = 2048
cfg.use_4bit = False                         # full LoRA (no QLoRA), better stability/quality
cfg.train.num_train_epochs = 50                  # high ceiling; early stopping decides actual stop
cfg.train.per_device_train_batch_size = 8        # safer: keeps effective batch near prior setup
cfg.train.per_device_eval_batch_size = 24
cfg.train.gradient_accumulation_steps = 1          # 8 * 1 ~= prior effective batch (4 * 2)
cfg.train.learning_rate = 8e-5
cfg.train.logging_steps = 5

# Eval/save once per epoch (lighter overhead)
cfg.train.eval_strategy = "epoch"
cfg.train.eval_steps = None
cfg.train.save_strategy = "epoch"
cfg.train.save_steps = None
cfg.train.save_total_limit = 20              # keep enough resume checkpoints while training

# Stop when validation loss no longer improves
cfg.train.early_stopping_patience = 3
cfg.train.early_stopping_threshold = 0.0

# Throughput-focused settings
cfg.train.dataloader_num_workers = 8
cfg.train.dataloader_pin_memory = True
cfg.train.auto_find_batch_size = True
cfg.train.tf32 = True
cfg.train.gradient_checkpointing = False

# --- Partial LoRA: freeze lower 30%, train top 70% of layers ---
cfg.lora_cfg["freeze_ratio"] = 0.30

# --- Generation/eval config ---
cfg.generation.max_new_tokens = 120
cfg.generation.batch_size = cfg.train.per_device_eval_batch_size

# --- W&B ---
cfg.wandb.enabled = True
cfg.wandb.project = "abstract-evaluator-qwen3-sft"
cfg.wandb.entity = None
cfg.wandb.tags = ["qwen3", "lora", "sft", "modular", "partial-lora-top70"]

cfg.ensure_dirs()
cfg.as_dict()









{'seed': 3407,
 'model_name': 'Qwen/Qwen3-8B',
 'run_name': 'qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70',
 'max_seq_length': 2048,
 'use_4bit': False,
 'output_root': '/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft',
 'data_paths': {'train_path': '/home/MohammadNabulsi/Essay Evaluator/data/data/train/all.jsonl',
  'val_path': '/home/MohammadNabulsi/Essay Evaluator/data/data/val/all.jsonl',
  'test_path': '/home/MohammadNabulsi/Essay Evaluator/data/data/test/all.jsonl',
  'combined_path': None},
 'train': {'num_train_epochs': 50,
  'per_device_train_batch_size': 8,
  'per_device_eval_batch_size': 24,
  'gradient_accumulation_steps': 1,
  'learning_rate': 8e-05,
  'warmup_ratio': 0.03,
  'weight_decay': 0.01,
  'lr_scheduler_type': 'cosine',
  'logging_steps': 5,
  'save_total_limit': 20,
  'early_stopping_patience': 3,
  'early_stopping_threshold': 0.0,
  'eval_strategy': 'epoch',
  'eval_steps': None,
  'save_strategy': 'epoch

In [4]:
set_global_seed(cfg.seed)
configure_wandb_dir(str(cfg.wandb.dir))

log_dir = cfg.output_root / "logs"
logger = setup_logger(
    name=f"{cfg.run_name}_pipeline",
    log_dir=log_dir,
    log_file=f"{cfg.run_name}.log",
)
logger.info("Initialized run config: %s", json.dumps(cfg.as_dict(), ensure_ascii=False))
log_dir


2026-05-26 21:02:42 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Initialized run config: {"seed": 3407, "model_name": "Qwen/Qwen3-8B", "run_name": "qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70", "max_seq_length": 2048, "use_4bit": false, "output_root": "/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft", "data_paths": {"train_path": "/home/MohammadNabulsi/Essay Evaluator/data/data/train/all.jsonl", "val_path": "/home/MohammadNabulsi/Essay Evaluator/data/data/val/all.jsonl", "test_path": "/home/MohammadNabulsi/Essay Evaluator/data/data/test/all.jsonl", "combined_path": null}, "train": {"num_train_epochs": 50, "per_device_train_batch_size": 8, "per_device_eval_batch_size": 24, "gradient_accumulation_steps": 1, "learning_rate": 8e-05, "warmup_ratio": 0.03, "weight_decay": 0.01, "lr_scheduler_type": "cosine", "logging_steps": 5, "save_total_limit": 20, "early_stopping_patience": 3, "early_stopping_th

PosixPath('/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/logs')

In [5]:
train_df, val_df, test_df = load_train_val_test_dfs(
    train_path=cfg.data_paths.train_path,
    val_path=cfg.data_paths.val_path,
    test_path=cfg.data_paths.test_path,
)

train_df, val_df, test_df = clean_train_val_test(train_df, val_df, test_df)

print("Shapes:", train_df.shape, val_df.shape, test_df.shape)
print("Train score dist:", score_distribution(train_df))
print("Val score dist:", score_distribution(val_df))
print("Test score dist:", score_distribution(test_df))


Shapes: (3055, 42) (298, 42) (298, 42)
Train score dist: {0: 0.10605564648117839, 1: 0.1656301145662848, 2: 0.3509001636661211, 3: 0.22585924713584288, 4: 0.15155482815057283}
Val score dist: {0: 0.10067114093959731, 1: 0.174496644295302, 2: 0.348993288590604, 3: 0.22483221476510068, 4: 0.15100671140939598}
Test score dist: {0: 0.10067114093959731, 1: 0.17114093959731544, 2: 0.348993288590604, 3: 0.22818791946308725, 4: 0.15100671140939598}


In [6]:
train_df = add_messages_and_targets(train_df)
val_df = add_messages_and_targets(val_df)
test_df = add_messages_and_targets(test_df)

print(json.dumps(train_df.iloc[0]["messages"], indent=2, ensure_ascii=False)[:2000])


[
  {
    "role": "system",
    "content": "You are a strict research abstract evaluator. You return only valid JSON."
  },
  {
    "role": "user",
    "content": "/no_think\nTask:\nEvaluate the quality of the following research abstract for conference acceptance.\n\nReference:\nA strong research abstract clearly presents the problem, methodology, contribution, and experimental evidence.\n\nRubric:\nScore scale:\n0 = Very poor abstract: missing most core components, unclear, generic, or unusable.\n1 = Weak abstract: contains a few useful elements but major components are missing or vague.\n2 = Borderline abstract: understandable but incomplete; some important components are weak or missing.\n3 = Good abstract: mostly complete, clear, and logically structured, with minor weaknesses.\n4 = Excellent abstract: complete, clear, concise, well-structured, and strongly communicates the paper's contribution and evidence.\n\nCriteria:\n10_specificity_and_evidence: Avoids generic claims and suppo

In [7]:
jsonl_paths = export_split_jsonl(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    output_dir=cfg.output_root / "jsonl",
)

ds = to_hf_dataset_dict(train_df, val_df, test_df)

print("JSONL paths:", jsonl_paths)
print(ds)


JSONL paths: {'train_jsonl': '/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/jsonl/train.jsonl', 'validation_jsonl': '/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/jsonl/validation.jsonl', 'test_jsonl': '/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/jsonl/test.jsonl'}
DatasetDict({
    train: Dataset({
        features: ['id', 'paper_id', 'messages', 'score', 'rationale', 'target_json'],
        num_rows: 3055
    })
    validation: Dataset({
        features: ['id', 'paper_id', 'messages', 'score', 'rationale', 'target_json'],
        num_rows: 298
    })
    test: Dataset({
        features: ['id', 'paper_id', 'messages', 'score', 'rationale', 'target_json'],
        num_rows: 298
    })
})


In [8]:
# Optional: token-length diagnostics (loads base tokenizer/model)
RUN_TOKEN_STATS = False

if RUN_TOKEN_STATS:
    lens = estimate_qwen_token_percentiles(
        model_name=cfg.model_name,
        max_seq_length=cfg.max_seq_length,
        df=pd.concat([train_df, val_df, test_df], ignore_index=True),
    )
    print(lens)


In [11]:
# Training resume plan:
# 1) Resume exactly from epoch-2 trainer checkpoint.
# 2) During continuation: save epoch adapters every 3 epochs.
# 3) During continuation: run generation-based eval every 3 epochs on VALIDATION only with BERTScore.
RUN_TRAINING = True
RESUME_FROM_EPOCH2 = False

output_dir = cfg.output_root / "models" / cfg.run_name
resume_checkpoint = output_dir / "checkpoint-764" if RESUME_FROM_EPOCH2 else None

print({"train_rows": len(train_df), "val_rows": len(val_df), "test_rows": len(test_df)})
if resume_checkpoint is not None:
    print("Resume checkpoint:", resume_checkpoint)
    if not resume_checkpoint.exists():
        raise FileNotFoundError(f"Epoch-2 checkpoint not found: {resume_checkpoint}")

if RUN_TRAINING:
    run_info = train_qwen3(
        cfg=cfg,
        ds=ds,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        include_bertscore_for_epoch_eval=True,
        run_epoch_generation_eval=True,
        run_epoch_test_eval=False,
        checkpoint_every_n_epochs=3,
        generation_eval_every_n_epochs=3,
        resume_from_checkpoint=resume_checkpoint,
        logger=logger,
    )
else:
    print("RUN_TRAINING=False -> evaluation mode only.")
    run_info = {
        "model_name": cfg.model_name,
        "run_name": cfg.run_name,
        "output_dir": str(output_dir),
        "adapter_dir": str(output_dir / "best_adapter"),
        "epoch_adapter_dir": str(output_dir / "epoch_adapters"),
        "eval_dir": str(cfg.output_root / "eval" / cfg.run_name),
    }

run_info



wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/MohammadNabulsi/.netrc.


{'train_rows': 3055, 'val_rows': 298, 'test_rows': 298}


wandb: Currently logged in as: s12217457 (s12217457-an-najah-national-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

Epoch,Training Loss,Validation Loss
1,0.811500,0.883702
2,0.774600,0.850093
3,0.724900,0.842183
4,0.728800,0.845248
5,0.636100,0.858288
6,0.609600,0.885088


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
***** train metrics *****
  epoch                    =          6.0
  freeze_cutoff            =           10
  freeze_frozen_params     =   8202859520
  freeze_num_layers        =           36
  freeze_total_params      = 8234382336.0
  freeze_trainable_params  =   31522816.0
  freeze_trainable_pct     =       0.3828
  total_flos               =  568361460GF
  train_loss               =       0.8815
  train_minutes            =      58.2837
  train_runtime            =   0:58:13.03
  train_samples_per_second =        43.73
  train_steps_per_second   =        5.468


eval/loss,█▂▁▂▄█
eval/runtime,█▂▃▄▄▁
eval/samples_per_second,▁▇▆▅▅█
eval/steps_per_second,▁▇▆▅▆█
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇████
train/global_step,▁▁▁▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇███
train/grad_norm,▁▂▂▁▁▂▃▁▃▂▂▂▃▃▃▃▄▄▃▄▅▄▄▅▅▅▅▆▅▅▅▆▅█▇█▇▆██
train/learning_rate,▁▁▂▃▃▃▅▆▇▇██████████████████████████████
train/loss,█▆▆▅▄▄▆▄▆▅▆▆▄▄▅▄▃▅▄▃▃▄▄▂▄▅▄▃▂▃▂▁▁▂▁▂▂▂▅▂
train_minutes,▁
eval/loss,0.88509


2026-05-26 22:02:07 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Training completed. Best adapter at /home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/models/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70/best_adapter


{'model_name': 'Qwen/Qwen3-8B',
 'run_name': 'qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70',
 'output_dir': '/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/models/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70',
 'adapter_dir': '/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/models/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70/best_adapter',
 'epoch_adapter_dir': '/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/models/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70/epoch_adapters',
 'eval_dir': '/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/eval/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70',
 'resumed_from': None}

In [12]:
# Compare BASE (untuned) vs BEST CHECKPOINT SO FAR (tuned)
# using DeBERTa for BERTScore (no retraining)

import numpy as np
import experiments.qwen.utils.pipeline as pipeline_mod
from experiments.utils.evaluation import (
    _get_bertscore,
    compute_eval_metrics as _orig_compute_eval_metrics,
)

BERTSCORE_MODEL_TYPE = "microsoft/deberta-xlarge-mnli"
BERTSCORE_BATCH_SIZE = 16
BERTSCORE_DEVICE = "cuda"  # change to "cpu" if needed

def _compute_eval_metrics_deberta(pred_df, include_bertscore=False):
    out = _orig_compute_eval_metrics(pred_df, include_bertscore=False)  # keep json/score/rouge/bleu
    if include_bertscore:
        preds = pred_df["pred_rationale"].fillna("").astype(str).tolist()
        refs = pred_df["rationale"].fillna("").astype(str).tolist()
        bert = _get_bertscore().compute(
            predictions=preds,
            references=refs,
            model_type=BERTSCORE_MODEL_TYPE,
            batch_size=BERTSCORE_BATCH_SIZE,
            device=BERTSCORE_DEVICE,
        )
        out["bertscore_precision"] = float(np.mean(bert["precision"]))
        out["bertscore_recall"] = float(np.mean(bert["recall"]))
        out["bertscore_f1"] = float(np.mean(bert["f1"]))
    return out

RUN_COMPARE_BASE_VS_BEST = True

if RUN_COMPARE_BASE_VS_BEST:
    output_dir = cfg.output_root / "models" / cfg.run_name
    ckpts = sorted(
        [p for p in output_dir.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found in: {output_dir}")

    latest_ckpt = ckpts[-1]
    trainer_state_path = latest_ckpt / "trainer_state.json"
    if not trainer_state_path.exists():
        raise FileNotFoundError(f"Missing trainer_state.json: {trainer_state_path}")

    state = json.loads(trainer_state_path.read_text(encoding="utf-8"))
    best_ckpt_str = state.get("best_model_checkpoint")
    if not best_ckpt_str:
        raise ValueError("best_model_checkpoint is missing in trainer_state.json")

    best_ckpt = Path(best_ckpt_str)
    if not best_ckpt.exists():
        raise FileNotFoundError(f"best_model_checkpoint path does not exist: {best_ckpt}")

    print("Latest checkpoint:", latest_ckpt)
    print("Best checkpoint:", best_ckpt)
    print("Best eval_loss:", state.get("best_metric"))
    print("BERTScore model:", BERTSCORE_MODEL_TYPE)

    prev_wandb = cfg.wandb.enabled
    old_compute = pipeline_mod.compute_eval_metrics
    cfg.wandb.enabled = False
    pipeline_mod.compute_eval_metrics = _compute_eval_metrics_deberta

    try:
        base_metrics = evaluate_base_model(
            cfg=cfg,
            val_df=val_df,
            test_df=test_df,
            include_bertscore=True,
            logger=logger,
        )

        best_metrics = evaluate_single_adapter(
            cfg=cfg,
            adapter_dir=best_ckpt,
            tag=f"best_ckpt_{best_ckpt.name}",
            val_df=val_df,
            test_df=test_df,
            include_bertscore=True,
            use_wandb=False,
            logger=logger,
        )
    finally:
        pipeline_mod.compute_eval_metrics = old_compute
        cfg.wandb.enabled = prev_wandb

    comparison_df = pd.DataFrame([base_metrics, best_metrics])
    display(comparison_df)

Latest checkpoint: /home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/models/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70/checkpoint-2292
Best checkpoint: /home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/models/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70/checkpoint-1146
Best eval_loss: 0.8421828746795654
BERTScore model: microsoft/deberta-xlarge-mnli
==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


2026-05-26 22:02:30 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generating validation predictions for base model
2026-05-26 22:02:37 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 0-24/298
2026-05-26 22:02:43 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 24-48/298
2026-05-26 22:02:50 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 48-72/298
2026-05-26 22:02:57 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 72-96/298
2026-05-26 22:03:04 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 96-120/298
2026-05-26 22:03:12 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 120-144/298
2026-05-26 22:03:19 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_to

==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


2026-05-26 22:06:14 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generating validation predictions for best_ckpt_checkpoint-1146
2026-05-26 22:06:25 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 0-24/298
2026-05-26 22:06:36 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 24-48/298
2026-05-26 22:06:49 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 48-72/298
2026-05-26 22:06:59 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 72-96/298
2026-05-26 22:07:09 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 96-120/298
2026-05-26 22:07:21 | INFO | qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70_pipeline | Generated batch 120-144/298
2026-05-26 22:07:32 | INFO | qwen3_8b_abstract_evaluator_lora_speed_mod

,run_name,model_name,checkpoint_tag,validation/json_parse_rate,validation/score_accuracy,validation/score_mae,validation/score_within_1_accuracy,validation/rouge_rouge1,validation/rouge_rouge2,validation/rouge_rougeL,...,test/score_mae,test/score_within_1_accuracy,test/rouge_rouge1,test/rouge_rouge2,test/rouge_rougeL,test/rouge_rougeLsum,test/bleu,test/bertscore_precision,test/bertscore_recall,test/bertscore_f1
0,qwen3_8b_abstract_evaluator_lora_speed_modular...,Qwen/Qwen3-8B,base_no_finetune,1.0,0.261745,1.040268,0.761745,0.347201,0.080971,0.220453,...,1.043624,0.761745,0.353601,0.081174,0.219375,0.219486,0.041602,0.639942,0.660585,0.649558
1,qwen3_8b_abstract_evaluator_lora_speed_modular...,Qwen/Qwen3-8B,best_ckpt_checkpoint-1146,1.0,0.483221,0.597315,0.919463,0.418161,0.128076,0.285244,...,0.577181,0.926174,0.428089,0.135559,0.290301,0.290358,0.095411,0.717247,0.703185,0.709868


In [13]:
comparison_df.columns

Index(['run_name', 'model_name', 'checkpoint_tag',
       'validation/json_parse_rate', 'validation/score_accuracy',
       'validation/score_mae', 'validation/score_within_1_accuracy',
       'validation/rouge_rouge1', 'validation/rouge_rouge2',
       'validation/rouge_rougeL', 'validation/rouge_rougeLsum',
       'validation/bleu', 'validation/bertscore_precision',
       'validation/bertscore_recall', 'validation/bertscore_f1',
       'test/json_parse_rate', 'test/score_accuracy', 'test/score_mae',
       'test/score_within_1_accuracy', 'test/rouge_rouge1',
       'test/rouge_rouge2', 'test/rouge_rougeL', 'test/rouge_rougeLsum',
       'test/bleu', 'test/bertscore_precision', 'test/bertscore_recall',
       'test/bertscore_f1'],
      dtype='str')

In [14]:
comparison_df[['test/bertscore_f1']]

,test/bertscore_f1
0,0.649558
1,0.709868


`Bertscore usign bert instead of deberta`

In [15]:
# # Compare BASE (untuned) vs BEST CHECKPOINT SO FAR (tuned)
# # - auto-detects best checkpoint from latest trainer_state.json
# # - evaluates both on val + test with BERTScore
# RUN_COMPARE_BASE_VS_BEST = True

# if RUN_COMPARE_BASE_VS_BEST:
#     output_dir = cfg.output_root / "models" / cfg.run_name
#     ckpts = sorted(
#         [p for p in output_dir.glob("checkpoint-*") if p.is_dir()],
#         key=lambda p: int(p.name.split("-")[-1]),
#     )
#     if not ckpts:
#         raise FileNotFoundError(f"No checkpoints found in: {output_dir}")

#     latest_ckpt = ckpts[-1]
#     trainer_state_path = latest_ckpt / "trainer_state.json"
#     if not trainer_state_path.exists():
#         raise FileNotFoundError(f"Missing trainer_state.json: {trainer_state_path}")

#     state = json.loads(trainer_state_path.read_text(encoding="utf-8"))
#     best_ckpt_str = state.get("best_model_checkpoint")
#     if not best_ckpt_str:
#         raise ValueError("best_model_checkpoint is missing in trainer_state.json")

#     best_ckpt = Path(best_ckpt_str)
#     if not best_ckpt.exists():
#         raise FileNotFoundError(f"best_model_checkpoint path does not exist: {best_ckpt}")

#     print("Latest checkpoint:", latest_ckpt)
#     print("Best checkpoint:", best_ckpt)
#     print("Best eval_loss:", state.get("best_metric"))

#     prev_wandb = cfg.wandb.enabled
#     cfg.wandb.enabled = False  # avoid wandb re-init issues during evaluation

#     try:
#         base_metrics = evaluate_base_model(
#             cfg=cfg,
#             val_df=val_df,
#             test_df=test_df,
#             include_bertscore=True,
#             logger=logger,
#         )

#         best_metrics = evaluate_single_adapter(
#             cfg=cfg,
#             adapter_dir=best_ckpt,
#             tag=f"best_ckpt_{best_ckpt.name}",
#             val_df=val_df,
#             test_df=test_df,
#             include_bertscore=True,
#             use_wandb=False,
#             logger=logger,
#         )
#     finally:
#         cfg.wandb.enabled = prev_wandb

#     comparison_df = pd.DataFrame([base_metrics, best_metrics])
#     display(comparison_df)



In [16]:
comparison_df[['test/score_accuracy', 'test/score_within_1_accuracy']]

,test/score_accuracy,test/score_within_1_accuracy
0,0.258389,0.761745
1,0.496644,0.926174


In [17]:
# Separate generation step (model-agnostic utility)
# This demonstrates using generate_predictions_from_messages independently.
RUN_GENERATION_STEP_ONLY = False

if RUN_GENERATION_STEP_ONLY:
    from experiments.qwen.utils.chat import make_inference_prompt

    adapter_dir = Path(run_info["adapter_dir"])
    model, tokenizer = load_qwen3_model_for_inference(
        model_name=cfg.model_name,
        adapter_dir=adapter_dir if adapter_dir.exists() else None,
        max_seq_length=cfg.max_seq_length,
    )

    demo_pred = generate_predictions_from_messages(
        eval_df=val_df.head(8),
        model=model,
        tokenizer=tokenizer,
        make_inference_prompt_fn=make_inference_prompt,
        parse_score_fn=parse_score,
        parse_rationale_fn=parse_rationale,
        max_seq_length=cfg.max_seq_length,
        max_new_tokens=cfg.generation.max_new_tokens,
        batch_size=4,
    )

    demo_pred.head()


In [18]:
# Save this compare run separately (no overwrite)
from datetime import datetime, timezone
from pathlib import Path
import json
import shutil

if "comparison_df" not in globals():
    raise RuntimeError("comparison_df is missing. Run the compare cell first.")

run_ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
run_label = f"deberta_xlarge_mnli_{run_ts}"
save_dir = cfg.output_root / "eval" / cfg.run_name / "separate_compare_runs" / run_label
save_dir.mkdir(parents=True, exist_ok=True)

# 1) Save comparison table
comparison_df.to_csv(save_dir / "comparison_df.csv", index=False)
comparison_df.to_json(save_dir / "comparison_df.json", orient="records", indent=2)

# 2) Save metadata
meta = {
    "timestamp_utc": run_ts,
    "run_name": cfg.run_name,
    "bertscore_model_type": globals().get("BERTSCORE_MODEL_TYPE", "unknown"),
    "latest_checkpoint": str(globals().get("latest_ckpt", "")),
    "best_checkpoint": str(globals().get("best_ckpt", "")),
    "best_eval_loss": (
        float(state.get("best_metric"))
        if ("state" in globals() and state.get("best_metric") is not None)
        else None
    ),
}
(save_dir / "run_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

# 3) Copy produced eval artifacts into this separate folder
main_eval_dir = cfg.output_root / "eval" / cfg.run_name
base_eval_dir = cfg.output_root / "eval" / f"{cfg.run_name}_base_no_finetune"
tag = f"best_ckpt_{best_ckpt.name}" if "best_ckpt" in globals() else None

candidates = [
    main_eval_dir / f"metrics_{tag}.json" if tag else None,
    main_eval_dir / f"validation_predictions_{tag}.csv" if tag else None,
    main_eval_dir / f"test_predictions_{tag}.csv" if tag else None,
    base_eval_dir / "metrics_base_no_finetune.json",
    base_eval_dir / "validation_predictions_base_no_finetune.csv",
    base_eval_dir / "test_predictions_base_no_finetune.csv",
]

copied = []
for src in candidates:
    if src is not None and src.exists():
        dst = save_dir / src.name
        shutil.copy2(src, dst)
        copied.append(str(dst))

print("Saved separate run to:", save_dir)
print("Files:")
for p in copied + [str(save_dir / "comparison_df.csv"), str(save_dir / "comparison_df.json"), str(save_dir / "run_meta.json")]:
    print(" -", p)

Saved separate run to: /home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/eval/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70/separate_compare_runs/deberta_xlarge_mnli_20260526_221137
Files:
 - /home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/eval/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70/separate_compare_runs/deberta_xlarge_mnli_20260526_221137/metrics_best_ckpt_checkpoint-1146.json
 - /home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/eval/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70/separate_compare_runs/deberta_xlarge_mnli_20260526_221137/validation_predictions_best_ckpt_checkpoint-1146.csv
 - /home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/abstract_evaluator_qwen3_sft/eval/qwen3_8b_abstract_evaluator_lora_speed_modular_partial_top70/separate_compare_runs/deberta_xlarge_mnli_20260526_221137/test_predictions_